# Phase 0 — Data Cleaning for DataCo Supply Chain
### Google Colab + Kaggle API Setup

**What this notebook does:**
- Connects to Kaggle using your kaggle.json API key
- Downloads the DataCo dataset directly into Colab
- Cleans all issues flagged by your ydata-profiling report
- Saves clean train/test files ready for AutoML

**Your dataset stats (from profiling report):**
- 180,519 rows · 53 columns · 3.5% missing · 47 alerts · 0 duplicates
- 17 categorical · 24 numeric · 9 text · 2 datetime · 1 unsupported

---
> **Run each cell one by one. Read the explanation before running.**

---
## CELL 1 — Install required libraries

**What:** Installs `kaggle` (to download the dataset) and `openpyxl` (just in case).  
**Why:** Google Colab doesn't have the Kaggle CLI pre-installed. We need it to pull the dataset directly from Kaggle without manually downloading and uploading files.  
**Expected output:** Lines of installation text ending with `Successfully installed kaggle-...`

In [ ]:
# Install Kaggle API
!pip install kaggle --quiet
print('✓ Kaggle installed')

---
## CELL 2 — Upload your kaggle.json API key

**What:** Opens a file upload dialog so you can upload your `kaggle.json` file.  
**Why:** Kaggle requires authentication to download datasets. The `kaggle.json` file contains your username and API key. You get it from: **Kaggle → Your Profile → Settings → API → Create New Token**.  
**Expected output:** A file picker appears. Select your `kaggle.json` and click Open.  
**Important:** This is temporary — Colab resets every session, so you need to re-upload kaggle.json each time you restart Colab.

In [ ]:
from google.colab import files

print('Please upload your kaggle.json file...')
uploaded = files.upload()   # a file picker dialog will appear

# Check that kaggle.json was uploaded
if 'kaggle.json' in uploaded:
    print('✓ kaggle.json uploaded successfully')
else:
    print('✗ ERROR: Make sure you uploaded the file named kaggle.json')

---
## CELL 3 — Move kaggle.json to the correct location and set permissions

**What:** Moves kaggle.json to `~/.kaggle/` directory and sets file permissions to 600.  
**Why:** The Kaggle CLI expects the API key to be at `~/.kaggle/kaggle.json` — it won't work from any other location. The `chmod 600` restricts access to your user only, which is a Kaggle requirement — without this, the API key is rejected.  
**Expected output:** No errors.

In [ ]:
import os

# Create the .kaggle directory if it doesn't exist
os.makedirs('/root/.kaggle', exist_ok=True)

# Move the uploaded file to the correct location
!cp kaggle.json /root/.kaggle/kaggle.json

# Set correct permissions — Kaggle requires this
!chmod 600 /root/.kaggle/kaggle.json

# Verify it's in the right place
!ls -la /root/.kaggle/
print('\n✓ kaggle.json is in the correct location with correct permissions')

---
## CELL 4 — Download the DataCo dataset from Kaggle

**What:** Uses the Kaggle CLI to download and unzip the DataCo Smart Supply Chain dataset.  
**Why:** Instead of manually downloading on your machine and uploading to Colab (slow and limited by Colab upload speed), we download directly from Kaggle's servers into Colab's runtime — much faster.  
**Expected output:** A progress bar showing download of ~65MB zip file, then unzipping.  
**Note:** The `-p /content/dataco` flag saves it to a folder named `dataco` inside Colab's `/content/` directory.

In [ ]:
# Download dataset — this is the exact Kaggle dataset URL slug
!kaggle datasets download -d shashwatwork/dataco-smart-supply-chain-for-big-data-analysis \
    -p /content/dataco --unzip

print('\nFiles downloaded:')
!ls -lh /content/dataco/

---
## CELL 5 — Import all libraries

**What:** Imports pandas, numpy, matplotlib, seaborn, sklearn.  
**Why:** These are the standard data science libraries. All are pre-installed in Colab so no pip install needed.  
**Expected output:** No errors. Just prints the versions for verification.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', 60)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

print(f'pandas  : {pd.__version__}')
print(f'numpy   : {np.__version__}')
print('✓ All libraries imported')

---
## CELL 6 — Load the main dataset

**What:** Reads `DataCoSupplyChainDataset.csv` into a pandas DataFrame.  
**Why `encoding='latin-1'`:** The CSV contains special characters (accented letters in customer/city names like São Paulo). The default UTF-8 encoding will crash with a `UnicodeDecodeError`. Latin-1 handles all these characters correctly.  
**Why `low_memory=False`:** With 53 columns and 180k rows, pandas sometimes reads a column as mixed types (int + string) if it scans only the first chunk. `low_memory=False` forces it to read the entire column before deciding the type — more memory usage but correct types.  
**Expected output:** Shape (180519, 53) and first 3 rows.

In [ ]:
FILE_PATH = '/content/dataco/DataCoSupplyChainDataset.csv'

df = pd.read_csv(
    FILE_PATH,
    encoding='latin-1',   # handles special characters — DO NOT change to utf-8
    low_memory=False       # prevents mixed-type column errors
)

# Store original shape for comparison at the end
original_rows, original_cols = df.shape

print(f'✓ Dataset loaded')
print(f'  Rows    : {original_rows:,}')
print(f'  Columns : {original_cols}')
print(f'  Memory  : {df.memory_usage(deep=True).sum() / 1e6:.1f} MB')
df.head(3)

---
## CELL 7 — Full initial inspection

**What:** Shows all 53 column names, their data types, and non-null counts.  
**Why:** Before cleaning anything, you need to see the full picture — which columns exist, what type pandas assigned them (object = text, int64/float64 = numeric), and how many non-null values each has. This confirms the profiling report findings in your actual data.

In [ ]:
print('All columns and dtypes:')
print('=' * 55)
info_df = pd.DataFrame({
    'dtype'     : df.dtypes,
    'non_null'  : df.notnull().sum(),
    'null_count': df.isnull().sum(),
    'null_pct'  : (df.isnull().sum() / len(df) * 100).round(1),
    'n_unique'  : df.nunique()
})
print(info_df.to_string())

---
## CELL 8 — Check target column distribution

**What:** Checks how balanced `Late_delivery_risk` (0 = on time, 1 = late) is.  
**Why:** Before any cleaning, you need to know your class distribution. If it's heavily imbalanced (e.g., 95% on time, 5% late), AutoML needs special handling (`balance_classes=True`). This check sets your expectations before training.

In [ ]:
TARGET = 'Late_delivery_risk'

print(f'Target column: {TARGET}')
print('=' * 40)
counts = df[TARGET].value_counts()
pcts   = df[TARGET].value_counts(normalize=True) * 100

for val in counts.index:
    label = 'ON TIME' if val == 0 else 'LATE'
    print(f'  {val} ({label:7s}): {counts[val]:6,} rows  ({pcts[val]:.1f}%)')

# Visualize
fig, ax = plt.subplots(figsize=(6, 3))
colors = ['#1D9E75', '#D85A30']
bars = ax.bar(['On Time (0)', 'Late (1)'], counts.values, color=colors, width=0.5)
for bar, pct in zip(bars, pcts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
            f'{pct:.1f}%', ha='center', va='bottom', fontsize=12, fontweight='bold')
ax.set_title('Target Distribution — Late_delivery_risk', fontsize=13)
ax.set_ylabel('Count')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.savefig('/content/target_distribution.png', dpi=120)
plt.show()

---
## CELL 9 — STEP 1: Drop constant columns

**What:** Finds and removes columns where every single row has the same value.  
**Why:** Your profiling report flagged:
- `Customer Email` → all rows say "XXXXXXXXX" (the company masked real emails)
- `Customer Password` → all rows say "XXXXXXXXX" (masked)
- `Product Status` → all rows are 0

A column with only one unique value has **zero variance**. Mathematically, it contributes nothing to any model — it can't distinguish between late and on-time orders because it looks the same for both. Keeping it only wastes compute time in AutoML.

We use `nunique() <= 1` to catch them automatically — this future-proofs the code in case there are other constant columns you haven't spotted.

In [ ]:
print('STEP 1 — Dropping constant columns')
print('=' * 55)

# Automatically find all constant columns
constant_cols = [col for col in df.columns if df[col].nunique() <= 1]

print(f'Constant columns found: {len(constant_cols)}')
for col in constant_cols:
    unique_val = df[col].unique()
    print(f'  ✗ {col!r:35s} → always = {unique_val}')

df.drop(columns=constant_cols, inplace=True)

print(f'\n✓ Dropped {len(constant_cols)} constant columns')
print(f'  Shape: {original_rows:,} × {original_cols} → {df.shape[0]:,} × {df.shape[1]}')

---
## CELL 10 — STEP 2: Drop 100% missing and high-missing columns

**What:** Removes columns where too much data is missing to be useful.  
**Why — from your profiling report:**
- `Product Description`: **100% missing** — 180,519 nulls out of 180,519 rows. The column exists in the schema but was never populated. Also flagged as "unsupported type". Absolutely drop it.
- `Order Zipcode`: **86.2% missing** — 155,679 missing rows. If you tried to impute this, you'd be inventing values for 86% of the column. That's not imputation — that's fabrication. The imputed values would be noise that misleads the model.

**Threshold = 40%:** Any column with more than 40% missing is dropped. This is a common industry standard. Below 40%, imputation is reasonable. Above 40%, you're making up too much data.

In [ ]:
print('STEP 2 — Dropping high-missing columns')
print('=' * 55)

MISSING_THRESHOLD = 40.0  # drop any column with > 40% missing

missing_pct = (df.isnull().sum() / len(df) * 100).round(2)

# Show ALL columns with any missing data first
has_missing = missing_pct[missing_pct > 0].sort_values(ascending=False)
print(f'Columns with missing values:')
for col, pct in has_missing.items():
    flag = ' ← DROPPING' if pct > MISSING_THRESHOLD else ''
    print(f'  {col:40s}: {pct:6.1f}%{flag}')

# Drop the high-missing ones
high_missing_cols = missing_pct[missing_pct > MISSING_THRESHOLD].index.tolist()
df.drop(columns=high_missing_cols, inplace=True)

print(f'\n✓ Dropped {len(high_missing_cols)} columns: {high_missing_cols}')
print(f'  Shape now: {df.shape[0]:,} × {df.shape[1]}')

---
## CELL 11 — STEP 3: Drop PII columns

**What:** Removes personally identifiable information columns.  
**Why:**
- `Customer Full Name` — a person's name has zero relationship to whether their shipment is late. It's random noise that could actually harm the model (model might memorize specific customers instead of learning patterns).
- `Product Image` — this is a URL string (e.g., `https://...jpg`). ML models cannot process image URLs as features. It would be treated as a high-cardinality text column with near-unique values per row, creating thousands of meaningless encoded columns.

**Note:** `Customer Email` and `Customer Password` were already removed in Step 1 (they were constant columns).

In [ ]:
print('STEP 3 — Dropping PII columns')
print('=' * 55)

pii_cols = [
    'Customer Full Name',  # name has no predictive power for delivery risk
    'Product Image',       # URL string — cannot be used as ML feature
]

# Only drop if they still exist (defensive check)
pii_to_drop = [c for c in pii_cols if c in df.columns]

for col in pii_to_drop:
    sample = df[col].dropna().iloc[0] if df[col].notnull().any() else 'N/A'
    print(f'  ✗ {col!r}: sample value = "{str(sample)[:60]}"')

df.drop(columns=pii_to_drop, inplace=True)

print(f'\n✓ Dropped {len(pii_to_drop)} PII columns')
print(f'  Shape now: {df.shape[0]:,} × {df.shape[1]}')

---
## CELL 12 — STEP 4: Drop data leakage columns ⚠️ MOST CRITICAL STEP

**What:** Removes columns that reveal the answer to the model before prediction time.  
**Why — this is the single most important cleaning step:**

Your profiling report showed:
- `Delivery Status` is **highly correlated** with `Late_delivery_risk`
- `Order Status` is **highly correlated** with `Delivery Status`

`Delivery Status` contains values like:
- **"Late delivery"** → means Late_delivery_risk = 1
- **"Advance shipping"** → means Late_delivery_risk = 0
- **"Shipping on time"** → means Late_delivery_risk = 0
- **"Shipping canceled"** → means Late_delivery_risk = 1

If the model sees `Delivery Status`, it literally reads the label off the column — this is why the Kaggle notebook claimed 99% accuracy. In real life, you're predicting disruptions **BEFORE** the shipment happens. At prediction time, `Delivery Status` would not exist yet — so it must never be in training data.

We first visualize the leakage to confirm it before dropping.

In [ ]:
print('STEP 4 — Proving data leakage before dropping')
print('=' * 55)

if 'Delivery Status' in df.columns:
    ct = pd.crosstab(
        df['Delivery Status'],
        df[TARGET],
        margins=True
    )
    ct.columns = ['On Time (0)', 'Late (1)', 'Total']
    print('Cross-tab: Delivery Status vs Late_delivery_risk')
    print(ct)
    print('\n→ See how each Delivery Status maps almost perfectly to one target value.')
    print('  This IS the target in disguise. It must be dropped.')

    # Visualize the leakage
    fig, ax = plt.subplots(figsize=(8, 4))
    crosstab_pct = pd.crosstab(
        df['Delivery Status'], df[TARGET], normalize='index'
    ) * 100
    crosstab_pct.columns = ['On Time (0)', 'Late (1)']
    crosstab_pct.plot(kind='bar', ax=ax, color=['#1D9E75', '#D85A30'],
                      edgecolor='white', width=0.6)
    ax.set_title('Data Leakage: Delivery Status perfectly predicts target', fontsize=12)
    ax.set_ylabel('% of rows')
    ax.set_xlabel('')
    ax.legend(title='Late_delivery_risk')
    plt.xticks(rotation=20, ha='right')
    plt.tight_layout()
    plt.savefig('/content/leakage_proof.png', dpi=120)
    plt.show()

In [ ]:
# Now drop both leakage columns
leakage_cols = [
    'Delivery Status',  # directly reveals Late_delivery_risk
    'Order Status',     # highly correlated with Delivery Status
]

leakage_to_drop = [c for c in leakage_cols if c in df.columns]
df.drop(columns=leakage_to_drop, inplace=True)

print(f'✓ Dropped leakage columns: {leakage_to_drop}')
print(f'  Shape now: {df.shape[0]:,} × {df.shape[1]}')

---
## CELL 13 — STEP 5: Drop redundant high-correlation columns

**What:** Removes columns that contain the same information as other columns already in the dataset.  
**Why:** Your profiling report found **35 high-correlation pairs**. When two columns say the same thing, keeping both causes **multicollinearity** — models get confused about which one matters, coefficients become unstable, and SHAP importance gets split between duplicate columns.

**Decisions from your report (keeping the better one from each pair):**
- `Category Name` already tells us the category → drop `Category Id`, `Product Category Id`, `Department Id`, `Department Name` (all encode the same grouping as numbers/text duplicates)
- `Order Item Discount Rate` is normalized (0–1) → drop `Order Item Discount` (raw dollar amount, same information)
- `Order Profit Per Order` → drop `Benefit per order` (same metric, different name)
- `Sales` already captures revenue → drop `Sales per customer` (correlated), `Order Item Total` (derivable)
- `Customer Country` captures geography → drop `Latitude`, `Longitude`, `Customer Zipcode` (all encode location)
- Pure ID columns: `Order Item Id` (uniform/unique — flagged by profiler), `Order Customer Id`, `Order Item Cardprod Id`, `Product Card Id` — row identifiers with no predictive signal
- `Order Id` and `Customer Id` — identifiers, not features

In [ ]:
print('STEP 5 — Dropping redundant high-correlation columns')
print('=' * 55)

redundant_cols = [
    # ── Category duplicates (keep Category Name) ──────────────────
    'Category Id',            # same grouping as Category Name but as int
    'Product Category Id',    # same as Category Id
    'Department Id',          # correlated with Category Id
    'Department Name',        # correlated with Category Name

    # ── Financial duplicates ───────────────────────────────────────
    'Order Item Discount',    # same info as Discount Rate (unnormalized)
    'Benefit per order',      # same as Order Profit Per Order
    'Sales per customer',     # correlated with Sales
    'Order Item Total',       # = Price × Quantity (derivable, redundant)

    # ── Geo duplicates (keep Customer Country) ─────────────────────
    'Latitude',               # correlated with Customer Country
    'Longitude',              # correlated with Customer Country

    # ── Pure ID columns (no predictive signal) ─────────────────────
    'Order Item Id',          # flagged as uniform + unique by profiler
    'Order Customer Id',      # just a join key, not a feature
    'Order Item Cardprod Id', # product ID — already have Category Name
    'Product Card Id',        # same as above
    'Order Id',               # transaction ID — not a feature
    'Customer Id',            # customer ID — not a feature
]

# Only drop columns that actually exist (defensive)
to_drop = [c for c in redundant_cols if c in df.columns]
not_found = [c for c in redundant_cols if c not in df.columns]

print(f'Dropping {len(to_drop)} redundant columns:')
for col in to_drop:
    print(f'  ✗ {col}')

if not_found:
    print(f'\nAlready gone (from previous steps): {not_found}')

df.drop(columns=to_drop, inplace=True)

print(f'\n✓ Dropped {len(to_drop)} redundant columns')
print(f'  Shape now: {df.shape[0]:,} × {df.shape[1]}')

---
## CELL 14 — STEP 6: Handle remaining missing values

**What:** Fills the remaining nulls that survived the column drops.  
**Why:**
- After dropping high-missing columns, the remaining nulls are small (< 5%) and safe to impute.
- **Numeric → median:** We use median (not mean) because shipping/financial data has outliers. A single extreme outlier would shift the mean significantly, causing bad imputation. Median is always the safer choice for skewed distributions.
- **Categorical → mode:** We fill missing categories with the most frequent value. This is the least disruptive choice — we're not inventing a new category, just using the most common existing one.

In [ ]:
print('STEP 6 — Handling remaining missing values')
print('=' * 55)

# Show what's still missing
still_missing = df.isnull().sum()
still_missing = still_missing[still_missing > 0].sort_values(ascending=False)

if len(still_missing) == 0:
    print('✓ No missing values remaining — nothing to fill!')
else:
    print(f'Columns still with missing values ({len(still_missing)}):')
    for col, cnt in still_missing.items():
        pct  = cnt / len(df) * 100
        kind = 'numeric' if df[col].dtype in ['int64','float64'] else 'categorical'
        strategy = 'median' if kind == 'numeric' else 'mode'
        print(f'  {col:40s}: {cnt:5,} ({pct:.1f}%) → fill with {strategy}')

    # Fill numeric columns with median
    numeric_cols = df.select_dtypes(include=['int64','float64']).columns
    for col in numeric_cols:
        if df[col].isnull().sum() > 0:
            val = df[col].median()
            df[col].fillna(val, inplace=True)

    # Fill categorical columns with mode
    object_cols = df.select_dtypes(include=['object']).columns
    for col in object_cols:
        if df[col].isnull().sum() > 0:
            val = df[col].mode()[0]
            df[col].fillna(val, inplace=True)

total_null = df.isnull().sum().sum()
print(f'\n✓ Total missing values remaining: {total_null}')
print(f'  Shape now: {df.shape[0]:,} × {df.shape[1]}')

---
## CELL 15 — STEP 7: Parse date columns and extract features

**What:** Converts raw date strings into usable numeric features.  
**Why:** Your profiling report detected **2 DateTime columns**. AutoML treats unparsed date strings as text (high-cardinality object columns) and gets zero useful signal from them. By extracting month, weekday, quarter, and year, we give the model real temporal patterns:

- `order_month` → captures seasonal effects (Q4 holiday = more orders = more delays)
- `order_quarter` → captures business cycle (fiscal quarters affect logistics volume)
- `order_weekday` → Mon–Fri have full warehouse staff; weekend orders often delayed
- `order_is_weekend` → direct binary flag for weekend ordering
- `ship_month`, `ship_weekday` → shipping date patterns separate from order date

After extraction, we drop the original date string columns — the model no longer needs them.

In [ ]:
print('STEP 7 — Parsing dates and extracting temporal features')
print('=' * 55)

ORDER_DATE_COL = 'order date (DateOrders)'
SHIP_DATE_COL  = 'shipping date (DateOrders)'

# ── Parse ORDER date ──────────────────────────────────────────────
if ORDER_DATE_COL in df.columns:
    df[ORDER_DATE_COL] = pd.to_datetime(df[ORDER_DATE_COL], errors='coerce')

    df['order_year']       = df[ORDER_DATE_COL].dt.year
    df['order_month']      = df[ORDER_DATE_COL].dt.month
    df['order_quarter']    = df[ORDER_DATE_COL].dt.quarter
    df['order_weekday']    = df[ORDER_DATE_COL].dt.dayofweek  # 0=Mon, 6=Sun
    df['order_day']        = df[ORDER_DATE_COL].dt.day
    df['order_is_weekend'] = (df['order_weekday'] >= 5).astype(int)

    print(f'✓ Extracted from order date:')
    print(f'  order_year, order_month, order_quarter, order_weekday, order_day, order_is_weekend')
    print(f'  Date range: {df[ORDER_DATE_COL].min().date()} to {df[ORDER_DATE_COL].max().date()}')

# ── Parse SHIPPING date ───────────────────────────────────────────
if SHIP_DATE_COL in df.columns:
    df[SHIP_DATE_COL] = pd.to_datetime(df[SHIP_DATE_COL], errors='coerce')

    df['ship_month']   = df[SHIP_DATE_COL].dt.month
    df['ship_weekday'] = df[SHIP_DATE_COL].dt.dayofweek
    df['ship_quarter'] = df[SHIP_DATE_COL].dt.quarter

    print(f'\n✓ Extracted from shipping date:')
    print(f'  ship_month, ship_weekday, ship_quarter')

# ── Drop original date string columns ─────────────────────────────
date_cols = [c for c in [ORDER_DATE_COL, SHIP_DATE_COL] if c in df.columns]
df.drop(columns=date_cols, inplace=True)
print(f'\n✓ Dropped original date columns (no longer needed): {date_cols}')
print(f'  Shape now: {df.shape[0]:,} × {df.shape[1]}')

---
## CELL 16 — STEP 8: Engineer new features

**What:** Creates brand new columns derived from existing ones.  
**Why — this is your most powerful step:**

Raw columns don't always directly capture what matters. You need to compute the *signal* yourself:

- **`delay_gap`**: `actual_days - scheduled_days`. Positive = late. Negative = early. Zero = on time. This is the **most direct measurement of disruption** in the entire dataset. AutoML cannot create this — it would need domain knowledge to know these two columns should be subtracted. This will be your #1 SHAP feature.
- **`delay_ratio`**: Normalized delay. A 2-day delay on a 3-day route (ratio = 0.67) is far more severe than a 2-day delay on a 15-day route (ratio = 0.13). Ratio captures severity better than raw days.
- **`is_delayed`**: Binary flag — simpler version of the target derived from delay_gap.
- **`profit_per_unit`**: High-value orders per unit may get priority treatment. Low-profit/unit orders may be deprioritized by logistics.
- **`discount_impact`**: Large discounts on high-sales items may indicate clearance/overstock — different logistics handling.

In [ ]:
print('STEP 8 — Engineering new features')
print('=' * 55)

REAL_DAYS  = 'Days for shipping (real)'
SCHED_DAYS = 'Days for shipment (scheduled)'

# ── delay_gap: core disruption signal ────────────────────────────
if REAL_DAYS in df.columns and SCHED_DAYS in df.columns:
    df['delay_gap'] = df[REAL_DAYS] - df[SCHED_DAYS]
    print(f'✓ delay_gap = actual_days - scheduled_days')
    print(f'  Min: {df["delay_gap"].min():.0f}  Max: {df["delay_gap"].max():.0f}')
    print(f'  Mean: {df["delay_gap"].mean():.2f}  (positive = on average late)')

    # ── delay_ratio: normalized severity ─────────────────────────
    df['delay_ratio'] = (
        df['delay_gap'] / df[SCHED_DAYS].replace(0, np.nan)
    ).fillna(0).round(4)
    print(f'\n✓ delay_ratio = delay_gap / scheduled_days (normalized severity)')

    # ── is_delayed: binary flag from gap ─────────────────────────
    df['is_delayed'] = (df['delay_gap'] > 0).astype(int)
    print(f'\n✓ is_delayed = 1 if delay_gap > 0 else 0')
    print(f'  Delayed orders: {df["is_delayed"].sum():,} ({df["is_delayed"].mean()*100:.1f}%)')

# ── profit_per_unit: order value density ─────────────────────────
if 'Order Profit Per Order' in df.columns and 'Order Item Quantity' in df.columns:
    df['profit_per_unit'] = (
        df['Order Profit Per Order'] /
        df['Order Item Quantity'].replace(0, np.nan)
    ).fillna(0).round(4)
    print(f'\n✓ profit_per_unit = profit / quantity')

# ── discount_impact: discount × sales ────────────────────────────
if 'Order Item Discount Rate' in df.columns and 'Sales' in df.columns:
    df['discount_impact'] = (df['Order Item Discount Rate'] * df['Sales']).round(4)
    print(f'✓ discount_impact = discount_rate × sales')

print(f'\n  Shape now: {df.shape[0]:,} × {df.shape[1]}')

# Visualize delay_gap distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['delay_gap'], bins=30, color='#1D9E75', edgecolor='white')
axes[0].axvline(0, color='red', linestyle='--', linewidth=2, label='On time (0)')
axes[0].set_title('delay_gap Distribution', fontsize=12)
axes[0].set_xlabel('delay_gap (days)')
axes[0].set_ylabel('Count')
axes[0].legend()

# delay_gap vs target
for val, label, color in [(0, 'On Time', '#1D9E75'), (1, 'Late', '#D85A30')]:
    subset = df[df[TARGET] == val]['delay_gap']
    axes[1].hist(subset, bins=30, alpha=0.6, label=label, color=color, edgecolor='white')
axes[1].set_title('delay_gap by Target Class', fontsize=12)
axes[1].set_xlabel('delay_gap (days)')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.tight_layout()
plt.savefig('/content/delay_gap_analysis.png', dpi=120)
plt.show()
print('→ See how delay_gap separates the two classes — this is your strongest feature!')

---
## CELL 17 — STEP 9: Encode categorical columns

**What:** Converts text/string columns into numbers.  
**Why:** All ML models (and AutoML frameworks) require numeric input. Text labels like "Standard Class" or "Central America" must become integers.

**Strategy:**
- **Low-cardinality (< 10 unique values) → One-Hot Encoding:** Creates a separate binary column for each category. Best when there are few categories and no ordinal relationship. Example: `Shipping Mode` (4 modes) → 4 binary columns.
- **High-cardinality (≥ 10 unique values) → Label Encoding:** Assigns an integer to each category. Used when there are many categories (country names, city names, product names) where one-hot would create hundreds of columns.

**Note for H2O AutoML:** H2O can handle categoricals natively — you can skip this step if you're going straight to H2O. But for PyCaret or sklearn-based scratch models, this step is required.

In [ ]:
print('STEP 9 — Encoding categorical columns')
print('=' * 55)

# Find all remaining object (text) columns
cat_cols = df.select_dtypes(include=['object']).columns.tolist()

if TARGET in cat_cols:
    cat_cols.remove(TARGET)   # never encode the target

print(f'Categorical columns to encode ({len(cat_cols)}):')
for col in cat_cols:
    print(f'  {col:40s}: {df[col].nunique()} unique values')

LOW_CARD_THRESHOLD = 10
low_card  = [c for c in cat_cols if df[c].nunique() < LOW_CARD_THRESHOLD]
high_card = [c for c in cat_cols if df[c].nunique() >= LOW_CARD_THRESHOLD]

print(f'\nLow-cardinality  → one-hot : {low_card}')
print(f'High-cardinality → label   : {high_card}')

In [ ]:
# One-hot encode low-cardinality columns
if low_card:
    df = pd.get_dummies(df, columns=low_card, drop_first=True, dtype=int)
    print(f'✓ One-hot encoded {len(low_card)} columns')
    print(f'  Shape now: {df.shape[0]:,} × {df.shape[1]}')

# Label encode high-cardinality columns
le = LabelEncoder()
encoded_map = {}  # save mappings so you can reverse later if needed

for col in high_card:
    if col in df.columns:
        df[col] = le.fit_transform(df[col].astype(str))
        encoded_map[col] = dict(zip(le.classes_, le.transform(le.classes_)))
        print(f'✓ Label encoded: {col} ({len(le.classes_)} unique classes)')

print(f'\nFinal shape: {df.shape[0]:,} × {df.shape[1]}')

# Confirm all columns are now numeric
non_numeric = df.select_dtypes(exclude=['int64','float64','int32',
                                        'float32','bool','uint8']).columns.tolist()
if TARGET in non_numeric:
    non_numeric.remove(TARGET)
if non_numeric:
    print(f'⚠ Still non-numeric: {non_numeric}')
else:
    print('✓ All feature columns are numeric')

---
## CELL 18 — STEP 10: Cap outliers

**What:** Clips extreme values in financial/numeric columns at reasonable boundaries.  
**Why:** Your profiling report flagged that `Days for shipping (real)` has 2.8% zeros, and financial columns have heavy tails (very large or very negative values).

**IQR method explained:**
- Q1 = 25th percentile, Q3 = 75th percentile
- IQR = Q3 − Q1 (the middle 50% of values)
- Lower bound = Q1 − 1.5 × IQR
- Upper bound = Q3 + 1.5 × IQR
- Any value outside these bounds gets clipped (not deleted)

**Why clip instead of delete?** Deleting outlier rows loses data (180k rows is large but you don't want to throw away legitimate orders). Clipping keeps all rows but reduces the extreme pull on model training.

**Why only for tree models later?** XGBoost and Random Forest are naturally robust to outliers — they split on thresholds, not magnitudes. Outlier capping matters more for the linear models you'll use in scratch training. We do it now so the cleaned file is ready for both.

In [ ]:
print('STEP 10 — Capping outliers with IQR method')
print('=' * 55)

def cap_iqr(df, col, factor=1.5):
    Q1    = df[col].quantile(0.25)
    Q3    = df[col].quantile(0.75)
    IQR   = Q3 - Q1
    lower = Q1 - factor * IQR
    upper = Q3 + factor * IQR
    n_out = ((df[col] < lower) | (df[col] > upper)).sum()
    df[col] = df[col].clip(lower=lower, upper=upper)
    return df, lower, upper, n_out

# Columns to cap — financial and delay features with known heavy tails
outlier_cols = [
    'Sales',
    'Order Profit Per Order',
    'Order Item Product Price',
    'Order Item Profit Ratio',
    'Order Item Quantity',
    'delay_gap',
    'delay_ratio',
    'profit_per_unit',
]

print(f'{"Column":40s} {"Outliers":>10s} {"New Range":>25s}')
print('-' * 80)
for col in outlier_cols:
    if col in df.columns:
        df, lo, hi, n_out = cap_iqr(df, col)
        print(f'{col:40s} {n_out:10,} {f"[{lo:.1f}, {hi:.1f}]":>25s}')

print(f'\n✓ Outlier capping complete')
print(f'  Shape now: {df.shape[0]:,} × {df.shape[1]}')

---
## CELL 19 — STEP 11: Time-based train/test split

**What:** Splits data chronologically — earliest 80% for training, latest 20% for testing.  
**Why NOT to use random split:** Your data spans multiple years. If you shuffle randomly:
- The model trains on orders from December 2017 and tests on orders from January 2015
- This is **future leaking into the past** — the model learns patterns from future data
- In real deployment, you will ALWAYS predict on future orders the model has never seen
- A time-based split honestly replicates this: trained on past, evaluated on future

**Sorting by `order_year` + `order_month`:** We sort by the temporal features we extracted in Step 7, ensuring the split follows the actual order timeline.

In [ ]:
print('STEP 11 — Time-based train/test split')
print('=' * 55)

SPLIT_RATIO = 0.80   # 80% train, 20% test

# Sort by temporal order (earliest → latest)
sort_cols = [c for c in ['order_year', 'order_month', 'order_day'] if c in df.columns]
df_sorted = df.sort_values(sort_cols).reset_index(drop=True)
print(f'✓ Sorted by: {sort_cols}')

# Split index
split_idx = int(len(df_sorted) * SPLIT_RATIO)

train_df = df_sorted.iloc[:split_idx].copy()
test_df  = df_sorted.iloc[split_idx:].copy()

# Separate features and target
X_train = train_df.drop(columns=[TARGET])
y_train = train_df[TARGET]
X_test  = test_df.drop(columns=[TARGET])
y_test  = test_df[TARGET]

print(f'\nTrain: {len(X_train):>7,} rows  ({len(X_train)/len(df)*100:.0f}%)')
print(f'Test:  {len(X_test):>7,} rows  ({len(X_test)/len(df)*100:.0f}%)')

print(f'\nTrain target distribution:')
tv = y_train.value_counts(normalize=True)
for v in sorted(tv.index):
    print(f'  {v} : {tv[v]*100:.1f}%')

print(f'\nTest target distribution:')
tv = y_test.value_counts(normalize=True)
for v in sorted(tv.index):
    print(f'  {v} : {tv[v]*100:.1f}%')

# Visualize temporal split
if 'order_year' in df_sorted.columns and 'order_month' in df_sorted.columns:
    fig, ax = plt.subplots(figsize=(10, 3))
    train_counts = train_df.groupby(['order_year','order_month']).size()
    test_counts  = test_df.groupby(['order_year','order_month']).size()

    ax.fill_between(range(len(train_counts)), train_counts.values,
                    alpha=0.7, color='#1D9E75', label=f'Train ({len(train_df):,} rows)')
    ax.fill_between(range(len(train_counts), len(train_counts) + len(test_counts)),
                    test_counts.values, alpha=0.7, color='#D85A30',
                    label=f'Test ({len(test_df):,} rows)')
    ax.axvline(len(train_counts), color='black', linestyle='--', linewidth=2)
    ax.set_title('Time-based Train/Test Split', fontsize=12)
    ax.set_xlabel('Month index (chronological)')
    ax.set_ylabel('Orders per month')
    ax.legend()
    plt.tight_layout()
    plt.savefig('/content/train_test_split.png', dpi=120)
    plt.show()

---
## CELL 20 — STEP 12: Final check and save

**What:** Runs a complete sanity check then saves 3 files.  
**Why save 3 files:**
- `dataco_train_clean.csv` → feed directly to AutoML (Phase 1)
- `dataco_test_clean.csv` → hold out completely until final evaluation
- `dataco_full_clean.csv` → use for scratch model experiments (Phase 2)

**Why check before saving:** Always verify no nulls remain, no non-numeric columns survived encoding, and the target column still has both classes — before you write the files and move on.

In [ ]:
print('STEP 12 — Final sanity check')
print('=' * 55)

# 1. Missing values
total_null = df.isnull().sum().sum()
status_null = '✓' if total_null == 0 else '✗'
print(f'{status_null} Missing values      : {total_null}')

# 2. Duplicates
total_dupes = df.duplicated().sum()
status_dupe = '✓' if total_dupes == 0 else '⚠'
print(f'{status_dupe} Duplicate rows      : {total_dupes:,}')

# 3. Non-numeric columns (excluding target)
non_num = [c for c in df.columns
           if df[c].dtype == 'object' and c != TARGET]
status_num = '✓' if len(non_num) == 0 else '✗'
print(f'{status_num} Non-numeric columns : {len(non_num)} {non_num if non_num else ""}')

# 4. Target column still valid
target_classes = df[TARGET].nunique()
status_tgt = '✓' if target_classes == 2 else '✗'
print(f'{status_tgt} Target classes      : {target_classes} (should be 2)')

# 5. Shape summary
print(f'\n  Original shape : {original_rows:,} × {original_cols}')
print(f'  Final shape    : {df.shape[0]:,} × {df.shape[1]}')
print(f'  Columns removed: {original_cols - df.shape[1]} net (added {df.shape[1] - original_cols + (original_cols - df.shape[1])} engineered features)')

# 6. Final feature list
features = [c for c in X_train.columns]
print(f'\n  Total features for AutoML: {len(features)}')
for i, col in enumerate(features):
    print(f'    {i+1:02d}. {col} ({X_train[col].dtype})')

In [ ]:
# Save all three output files
print('Saving cleaned datasets...')

train_df.to_csv('/content/dataco_train_clean.csv', index=False)
test_df.to_csv('/content/dataco_test_clean.csv',   index=False)
df_sorted.to_csv('/content/dataco_full_clean.csv', index=False)

print('✓ Saved: /content/dataco_train_clean.csv')
print('         → Use this file for Phase 1 AutoML training')
print('✓ Saved: /content/dataco_test_clean.csv')
print('         → Hold this out — use ONLY for final model evaluation')
print('✓ Saved: /content/dataco_full_clean.csv')
print('         → Use this for Phase 2 scratch model experiments')

# Download to your local machine (optional)
print('\nTo download the files to your computer, run the next cell.')

In [ ]:
# OPTIONAL — Download cleaned files to your local machine
# Uncomment the lines below if you want to save them locally

# from google.colab import files
# files.download('/content/dataco_train_clean.csv')
# files.download('/content/dataco_test_clean.csv')
# files.download('/content/dataco_full_clean.csv')

---
## CELL 21 — Correlation heatmap of final features

**What:** Plots a heatmap showing correlation between your most important features and the target.  
**Why:** This gives you a visual confirmation that your engineered features (`delay_gap`, `delay_ratio`) show strong correlation with `Late_delivery_risk`, while the remaining features show diverse, non-redundant patterns. This is your final quality check before handing data to AutoML.

In [ ]:
# Select key columns for the heatmap
key_cols = [
    'delay_gap', 'delay_ratio', 'is_delayed',
    'Days for shipping (real)', 'Days for shipment (scheduled)',
    'Order Profit Per Order', 'profit_per_unit',
    'Sales', 'Order Item Quantity',
    'order_month', 'order_weekday', 'order_is_weekend',
    TARGET
]
available = [c for c in key_cols if c in df.columns]

fig, ax = plt.subplots(figsize=(13, 9))
corr = df[available].corr()
mask = np.zeros_like(corr, dtype=bool)
mask[np.triu_indices_from(mask, k=1)] = True  # show lower triangle only

sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='RdYlGn', center=0,
    square=True, linewidths=0.5,
    cbar_kws={'shrink': 0.8},
    ax=ax
)
ax.set_title('Feature Correlation Heatmap (after cleaning)', fontsize=13, pad=15)
plt.tight_layout()
plt.savefig('/content/correlation_heatmap_final.png', dpi=130, bbox_inches='tight')
plt.show()
print('✓ Saved: /content/correlation_heatmap_final.png')
print('\nPhase 0 complete. Ready for Phase 1 — AutoML!')